In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_house_type_price_data(city_name):
    """
    获取指定城市的户型与单价原始明细数据。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        concat(room_num, '室', hall_num, '厅', bathroom_num, '卫') as room_type,
        unit_price
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND room_num IS NOT NULL 
      AND hall_num IS NOT NULL
      AND bathroom_num IS NOT NULL
      AND unit_price IS NOT NULL
      AND unit_price > 0
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_price_boxplot(city):
    """绘制热门户型单价箱线图"""
    df = fetch_house_type_price_data(city)
    
    if df.empty or len(df) < 50:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 有效明细数据不足，无法绘制分布图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 数据预处理与过滤 ---
    top_10_types = df['room_type'].value_counts().nlargest(10).index.tolist()
    plot_df = df[df['room_type'].isin(top_10_types)].copy()

    # 剔除单价最高的那 1% 的极端豪宅（异常点），防止箱体被压得太扁
    p99 = plot_df['unit_price'].quantile(0.99)
    plot_df = plot_df[plot_df['unit_price'] <= p99]

    # --- 2. 排序逻辑 ---
    type_medians = plot_df.groupby('room_type')['unit_price'].median().sort_values()
    sorted_types = type_medians.index.tolist()

    # --- 3. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.boxplot(
        x='room_type', 
        y='unit_price', 
        data=plot_df, 
        order=sorted_types,
        palette='Set3',
        width=0.5,
        linewidth=1.5,
        showfliers=True, 
        flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.3, 'markerfacecolor': 'gray'}
    )

    # --- 4. 添加基准线与修饰 ---
    # 计算这十大热门户型的总体单价中位数，作为全市的“平均水位线”
    city_median = plot_df['unit_price'].median()
    ax.axhline(city_median, color='#E74C3C', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(
        len(sorted_types) - 0.5, city_median + (p99 * 0.02), 
        f'全市热门户型中位单价: {city_median:,.0f} 元/㎡', 
        color='#E74C3C', fontweight='bold', ha='right', va='bottom', fontsize=11
    )

    plt.title(f'[{city}] TOP 10 热门户型单价波动与溢价分析 (按单价中位数排序)', fontsize=16, pad=20)
    plt.xlabel('热门户型 (从刚需低价到改善高价)', fontsize=12, labelpad=10)
    plt.ylabel('单价 (元/平米)', fontsize=12)
    
    # X轴标签轻微倾斜，避免文字拥挤
    plt.xticks(rotation=15, fontsize=11)
    
    # 辅助网格线，只保留Y轴横线
    ax.grid(axis='y', linestyle='-', alpha=0.4)
    ax.grid(axis='x', visible=False)
    sns.despine(left=True, top=True, right=True)

    plt.tight_layout()
    plt.show()

In [6]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_price_boxplot(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_price_boxplot(default_city)

In [7]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()